In [ ]:
# 1/6 라이브러리 로드
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
import os

BASE_DIR = r'C:\Users\seohg\OneDrive\바탕 화면\seoul_map'

print('라이브러리 로드 완료')

In [ ]:
# 2/6 브랜드 이력 로드 및 브랜드 매핑
# brand_history.csv: 카페 업종 필터링 후 브랜드 키워드로 추출한 데이터
# 컬럼: 상가업소번호, 상호명, 건물관리번호, 층정보, 경도, 위도, period 등

df = pd.read_csv(os.path.join(BASE_DIR, 'brand_history.csv'))

# 브랜드 키워드 → 대표 브랜드명 매핑
brand_map = {
    '투썸': '투썸플레이스',
    '이디야': '이디야',
    '빽다방': '빽다방',
    '공차': '공차',
    '할리스': '할리스',
    '파스쿠찌': '파스쿠찌',
    '카페베네': '카페베네',
    '엔제리너스': '엔제리너스',
    '탐앤탐': '탐앤탐스',
    '폴바셋': '폴바셋',
    '컴포즈': '컴포즈',
    '메가커피': '메가커피',
    '요거프레소': '요거프레소'
}

def get_brand(name):
    for kw, brand in brand_map.items():
        if kw in str(name):
            return brand
    return None

df['brand'] = df['상호명'].apply(get_brand)
df = df[df['brand'].notna()]

# period 숫자 변환 (df_201512 → 201512)
df['period_num'] = df['period'].str.replace('df_', '', regex=False).astype(int)

print(f'총 브랜드 이력: {len(df):,}행')
print(f'브랜드 수: {df["brand"].nunique()}개')
print()
print('브랜드별 행수:')
print(df['brand'].value_counts())

In [ ]:
# 3/6 생존기간 계산 및 라벨링
#
# 계산 구조:
# - 상가업소번호 기준으로 groupby
#   → 상가업소번호는 한 점포의 고유 ID
#   → 같은 번호가 몇 개 분기에 등장하는지 = 생존분기수
#
# - 생존 라벨 기준: 8분기(2년) 이상 = 생존(1), 미만 = 폐업(0)
#   → Chapter 1의 위험 기준(2년 이하 생존)과 동일한 기준 사용
#   → 일관성 확보

survival = df.groupby(['상가업소번호', 'brand']).agg(
    생존분기수=('period_num', 'nunique'),   # 등장한 분기 수
    입점분기=('period_num', 'min'),          # 첫 등장 분기
    퇴점분기=('period_num', 'max'),          # 마지막 등장 분기
    층정보=('층정보', 'first'),
    건물관리번호=('건물관리번호', 'first'),
    경도=('경도', 'first'),
    위도=('위도', 'first'),
    시군구명=('시군구명', 'first'),
    행정동명=('행정동명', 'first'),
    도로명주소=('도로명주소', 'first')
).reset_index()

# 생존 라벨: 8분기 이상 = 1(생존), 미만 = 0(폐업)
survival['생존여부'] = (survival['생존분기수'] >= 8).astype(int)

print('=== 생존 라벨 분포 ===')
print(survival['생존여부'].value_counts())
print(f'전체 생존율: {survival["생존여부"].mean():.1%}')
print()
print('=== 브랜드별 생존율 ===')
brand_survival = survival.groupby('brand')['생존여부'].agg(['mean','count'])
brand_survival.columns = ['생존율', '매장수']
print(brand_survival.sort_values('생존율', ascending=False).round(3))

In [ ]:
# 4/6 물리적 특성 결합
#
# 계산 구조:
# - 위치ID = 건물관리번호 + 층정보 + 'nan' (호정보 결측)
#   → Chapter 1과 동일한 위치ID 생성 방식
# - master_enriched.csv에서 물리적 특성 left join
#   (연면적, 건물지상층수, 지하철거리, 건물내위치수, 주변상가수, 상권유형, 도로유형)

survival['위치ID'] = (
    survival['건물관리번호'].astype(str) + '_' +
    survival['층정보'].astype(str) + '_nan'
)

master = pd.read_csv(
    os.path.join(BASE_DIR, 'viz_map_v2.csv'),
    usecols=['위치ID', '층구분', '연면적', '지하철거리_m',
             '건물내위치수', 'TRDAR_SE_1', '도로유형']
)

# 건물지상층수는 master_enriched에서
master_enriched = pd.read_csv(
    r'C:\Users\seohg\OneDrive\바탕 화면\2026\seoul datalob contest\master_enriched.csv',
    usecols=['위치ID', '건물지상층수', '주변상가수_200m']
)

merged = survival.merge(master, on='위치ID', how='left')
merged = merged.merge(master_enriched, on='위치ID', how='left')

matched = merged['층구분'].notna()
print(f'물리적 특성 매칭률: {matched.mean():.1%}')
print(f'매칭 수: {matched.sum():,} / {len(merged):,}')

model_df = merged.dropna(subset=['층구분']).copy()
print(f'모델 학습 데이터: {len(model_df):,}개')

In [ ]:
# 5/6 CatBoost 생존 분류 모델
#
# 계산 구조:
# - 종속변수: 생존여부 (1=생존, 0=폐업)
# - 독립변수: 위치 물리적 특성 7개 (Chapter 1과 동일)
# - 모델: CatBoost (범주형 변수 그대로 처리)
# - 목적: 어떤 물리적 조건에서 프랜차이즈 카페가 오래 버티는가
#
# Chapter 1과의 차이:
# - Chapter 1 라벨: 상권대비 교체율 상위 10% (위험 자리 식별)
# - Chapter 2 라벨: 브랜드 생존 2년 이상 (좋은 자리 식별)

cat_features = ['층구분', 'TRDAR_SE_1', '도로유형']
num_features = ['연면적', '건물지상층수', '지하철거리_m', '건물내위치수', '주변상가수_200m']
features = cat_features + num_features

X = model_df[features].fillna(-1)
y = model_df['생존여부']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    cat_features=cat_features,
    verbose=0,
    random_seed=42
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(f'ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}')
print()
print(classification_report(y_test, y_pred))
print()
print('=== Feature Importance ===')
fi = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
for k, v in fi.items():
    print(f'  {k}: {v:.1f}%')

In [ ]:
# 6/6 Chapter 1 vs Chapter 2 Feature Importance 비교
#
# 핵심 발견:
# - Chapter 1 (일반 소상공인): 층구분이 1위 → 자리 자체가 중요
# - Chapter 2 (프랜차이즈 카페): 주변상가수가 1위 → 상권 밀도가 중요
# → 소상공인과 프랜차이즈가 보는 입지 기준이 다르다

ch1_fi = {
    '층구분': 31.3,
    '연면적': 22.3,
    '건물내위치수': 13.8,
    '건물지상층수': 9.3,
    '주용도': 7.4,
    '주변상가수_200m': 4.7,
    '지하철거리_m': 3.8,
    'TRDAR_SE_1': 2.3,
    '도로유형': 1.1
}

ch2_fi = fi.to_dict()

compare = pd.DataFrame({
    'Chapter1_일반소상공인': ch1_fi,
    'Chapter2_프랜차이즈카페': ch2_fi
}).fillna(0).round(1)

print('=== Chapter 1 vs Chapter 2 Feature Importance 비교 ===')
print(compare.sort_values('Chapter1_일반소상공인', ascending=False).to_string())
print()
print('=== 해석 ===')
print('소상공인: 층구분(건물 자체 특성)이 가장 중요')
print('프랜차이즈: 주변상가수(상권 밀도)가 가장 중요')
print('→ 프랜차이즈는 이미 활성화된 상권을 선택한다')